# Đánh giá 2 dataset HF: Harvard-GF 96³ vs 128³ (classification)

So sánh trực tiếp 2 dataset đã push lên Hugging Face:
- `tqhuyen/harvard-oct-glaucoma-96`
- `tqhuyen/harvard-oct-glaucoma-128`

Chạy trên **Colab GPU (L4/T4/A100)**. Gồm 4 thí nghiệm (Ảnh / Đặc trưng / Nhiệm vụ / Batch-size) và
tự kết luận **dataset nào tốt hơn** cho bài toán phân loại.

**Lưu TẤT CẢ kết quả lên Drive**: figure, report JSON, CSV metric, model, và file tóm tắt
`RESULT_96_vs_128.md` -> `/content/drive/MyDrive/MasterBKDN/Thesis/resolution_96_128_hf[_figures]`.
Cuối notebook có ô để bạn tự ghi lại kết quả.

## 1. Setup

In [ ]:
!git clone --depth 1 https://github.com/Tqhuyen/glaucoma-thesis.git /content/glaucoma-thesis 2>/dev/null || git -C /content/glaucoma-thesis pull --ff-only 2>/dev/null || true
%cd /content/glaucoma-thesis
!pip install -q wandb scikit-learn scipy matplotlib huggingface_hub hf_transfer pandas
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import os, sys, json, time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, "scripts")
import resolution_study as rs
import compare_resolutions as cr

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Cần GPU: Runtime -> Change runtime type -> L4/T4/A100")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        print("no HF_TOKEN:", e)
if not HF_TOKEN:
    raise RuntimeError("Thiếu HF_TOKEN (Colab Secrets). Cả 2 repo dataset là private.")
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login, snapshot_download
login(token=HF_TOKEN)
print("HF login OK")


## 2. Cấu hình

In [ ]:
SEED = 42
SIZES = (96, 128)
HF_REPOS = {96: "tqhuyen/harvard-oct-glaucoma-96", 128: "tqhuyen/harvard-oct-glaucoma-128"}
DATA_CACHE = "/content/hf_datasets"
SPLIT = "Training"
VAL_SPLIT = "Validation"

TRAIN_N = 0
VAL_N = 0
EPOCHS = 30
BS = 4
BS_LARGE = 8
WIDTH = 24

RUN_EXP1 = True
RUN_EXP2 = True
RUN_EXP3 = True
RUN_EXP4 = True

np.random.seed(SEED)
torch.manual_seed(SEED)
FIG_DIR = os.path.join("figures", "resolution_hf")
os.makedirs(FIG_DIR, exist_ok=True)
DRIVE_ROOT = os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/MasterBKDN/Thesis")
DRIVE_FIG = os.path.join(DRIVE_ROOT, "resolution_96_128_hf_figures")
DRIVE_MODEL = os.path.join(DRIVE_ROOT, "resolution_96_128_hf")
print("config:", SIZES, "| epochs", EPOCHS, "| bs", BS, "bs_large", BS_LARGE, "| train_n", TRAIN_N)


## 3. Tải 2 dataset từ HF + nạp memmap

In [ ]:
DATA = {}
for s in SIZES:
    repo = HF_REPOS[s]
    local = os.path.join(DATA_CACHE, f"glaucoma_all_{s}")
    print(f"[hf] snapshot {repo} -> {local}", flush=True)
    snapshot_download(repo_id=repo, repo_type="dataset", local_dir=local)
    DATA[s] = {
        "train_v": np.load(os.path.join(local, f"{SPLIT}_volumes.npy"), mmap_mode="r"),
        "train_l": np.load(os.path.join(local, f"{SPLIT}_labels.npy")),
        "val_v": np.load(os.path.join(local, f"{VAL_SPLIT}_volumes.npy"), mmap_mode="r"),
        "val_l": np.load(os.path.join(local, f"{VAL_SPLIT}_labels.npy")),
    }
    print(f"[data] {s}^3 train {DATA[s]['train_v'].shape} val {DATA[s]['val_v'].shape}", flush=True)


## 4. Wandb + Drive helpers

In [ ]:
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception as e:
        print("no WANDB_API_KEY:", e)

RUN_NAME = "resolution_hf_96_128_" + time.strftime("%Y%m%d_%H%M%S")
run = None
if os.environ.get("WANDB_API_KEY"):
    try:
        import wandb
        run = wandb.init(project="glaucoma-thesis", name=RUN_NAME,
                         config={"sizes": list(SIZES), "epochs": EPOCHS, "bs": BS, "bs_large": BS_LARGE})
    except Exception as e:
        print("[wandb] init failed:", e)
RUN_WANDB = run is not None
print("wandb:", RUN_NAME if RUN_WANDB else None)


def mount_drive():
    if os.path.isdir(DRIVE_ROOT):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return os.path.isdir(DRIVE_ROOT)
    except Exception as e:
        print("[drive] mount skipped:", e)
        return False


DRIVE_READY = mount_drive()
print("drive:", DRIVE_READY, DRIVE_ROOT)


## 5. Thí nghiệm 1 — Reconstruction (128 → 96 → 128)

Tham chiếu là 128³ (bản 200³ không có sẵn); hạ 128→96 (Gaussian+trilinear) rồi nâng về 128, đo PSNR/SSIM.

In [ ]:
exp1 = {}
if RUN_EXP1:
    rv = DATA[128]["val_v"]
    n = min(50, len(rv))
    idx = np.random.default_rng(0).choice(len(rv), size=n, replace=False)
    ps, ss = [], []
    for j in idx:
        x = np.asarray(rv[j])
        x = (x[0] if x.ndim == 4 else x).astype(np.float32)
        up = rs.resize_volume(rs.downsample_volume(x, (96,) * 3, "gaussian_trilinear"), (128,) * 3)
        ps.append(rs.psnr(x, up)); ss.append(rs.ssim3d(x, up))
    exp1 = {"psnr": float(np.mean(ps)), "ssim": float(np.mean(ss))}
    print(f"[exp1] 96 vs 128: PSNR {exp1['psnr']:.2f} dB | SSIM {exp1['ssim']:.4f}")
    v = np.asarray(rv[idx[0]]); v = (v[0] if v.ndim == 4 else v).astype(np.float32)
    mid = 128 // 2
    up = rs.resize_volume(rs.downsample_volume(v, (96,) * 3, "gaussian_trilinear"), (128,) * 3)
    fig, axs = plt.subplots(1, 2, figsize=(9, 4.6))
    for ax, im, t in zip(axs, [v[:, :, mid], up[:, :, mid]], ["128^3 (ref)", "128 -> 96 -> 128"]):
        ax.imshow(im, cmap="gray", vmin=np.percentile(v, 1), vmax=np.percentile(v, 99)); ax.set_title(t); ax.axis("off")
    p1 = os.path.join(FIG_DIR, "exp1_128_to_96_roundtrip.png")
    fig.tight_layout(); fig.savefig(p1, dpi=160, bbox_inches="tight"); plt.close(fig)
    print("wrote", p1)
    if RUN_WANDB:
        import wandb
        run.log({"exp1/psnr": exp1["psnr"], "exp1/ssim": exp1["ssim"], "exp1/img": wandb.Image(p1)}, step=1)


## 6. Thí nghiệm 2 — Proxy Model Training (96³ vs 128³, cùng seed/hp)

In [ ]:
exp2, models, hists = {}, {}, {}
if RUN_EXP2:
    for s in SIZES:
        print(f"[exp2] train {s}^3 ...", flush=True)
        model, met, feats, yv, hist = cr.train_proxy(
            DATA[s]["train_v"], DATA[s]["train_l"], DATA[s]["val_v"], DATA[s]["val_l"],
            BS, EPOCHS, DEVICE, SEED, width=WIDTH, n_train=TRAIN_N, n_val=VAL_N)
        exp2[s], models[s], hists[s] = met, model, hist
        print(f"[exp2] {s}^3 AUC={met['auc_roc']:.4f} AP={met['auc_pr']:.4f} F1={met['f1']:.4f} "
              f"bAcc={met['balanced_acc']:.4f} ECE={met['ece']:.4f} ({met['train_min']} min)", flush=True)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.4))
    for s, h in hists.items():
        axs[0].plot([x["epoch"] for x in h], [x["loss"] for x in h], label=f"{s}^3")
        axs[1].plot([x["epoch"] for x in h], [x["auc_roc"] for x in h], label=f"{s}^3")
    axs[0].set_title("Val loss"); axs[0].set_xlabel("epoch")
    axs[1].set_title("Val AUC-ROC"); axs[1].set_xlabel("epoch")
    for a in axs:
        a.legend(); a.grid(alpha=0.3)
    p2 = os.path.join(FIG_DIR, "exp2_learning_curves.png")
    fig.tight_layout(); fig.savefig(p2, dpi=160, bbox_inches="tight"); plt.close(fig)
    print("wrote", p2)
    if RUN_WANDB:
        import wandb
        for s, met in exp2.items():
            run.log({f"exp2/{s}/{k}": v for k, v in met.items()}, step=2)
        run.log({"exp2/img": wandb.Image(p2)}, step=2)


## 7. Thí nghiệm 3 — CKA + Silhouette + t-SNE

In [ ]:
exp3 = {}
if RUN_EXP3 and models:
    feats = {}
    for s in SIZES:
        va = cr.VolumeDataset(DATA[s]["val_v"], DATA[s]["val_l"], n_max=VAL_N, seed=SEED + 1)
        from torch.utils.data import DataLoader
        dl = DataLoader(va, batch_size=BS, shuffle=False, num_workers=0)
        _, yv, f = cr.predict(models[s], dl, DEVICE)
        feats[s] = f
    cka = rs.linear_cka(feats[128], feats[96])
    from sklearn.manifold import TSNE
    emb, sil = {}, {}
    for s in SIZES:
        emb[s] = TSNE(n_components=2, perplexity=min(30, len(feats[s]) - 1), init="pca",
                      random_state=0).fit_transform(feats[s])
        sil[s] = rs.silhouette_np(cr.pca_np(feats[s], 16), yv)
    drop = (sil[128] - sil[96]) / sil[128] if sil[128] else float("nan")
    exp3 = {"cka": float(cka), "sil_128": float(sil[128]), "sil_96": float(sil[96]), "sil_drop": float(drop)}
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.8))
    for ax, s in zip(axs, SIZES):
        ax.scatter(emb[s][:, 0], emb[s][:, 1], c=yv, s=8, cmap="coolwarm", alpha=0.7)
        ax.set_title(f"{s}^3 | silhouette={sil[s]:.3f}"); ax.axis("off")
    fig.suptitle(f"CKA(128,96) = {cka:.3f}")
    p3 = os.path.join(FIG_DIR, "exp3_features_tsne.png")
    fig.tight_layout(); fig.savefig(p3, dpi=160, bbox_inches="tight"); plt.close(fig)
    print(f"[exp3] CKA={cka:.4f} sil128={sil[128]:.4f} sil96={sil[96]:.4f} drop={drop*100:.2f}%")
    print("wrote", p3)
    if RUN_WANDB:
        import wandb
        run.log({**{f"exp3/{k}": v for k, v in exp3.items()}, "exp3/img": wandb.Image(p3)}, step=3)


## 8. Thí nghiệm 4 — Batch-size Trade-off

128³ BS nhỏ (đã có ở Exp2) vs 96³ BS lớn.

In [ ]:
exp4 = {}
if RUN_EXP4 and 128 in exp2:
    print(f"[exp4] train 96^3 bs={BS_LARGE} ...", flush=True)
    _, m96l, _, _, _ = cr.train_proxy(
        DATA[96]["train_v"], DATA[96]["train_l"], DATA[96]["val_v"], DATA[96]["val_l"],
        BS_LARGE, EPOCHS, DEVICE, SEED, width=WIDTH, n_train=TRAIN_N, n_val=VAL_N)
    exp4 = {f"128_bs{BS}": exp2[128], f"96_bs{BS_LARGE}": m96l}
    print(f"[exp4] AUC 128 bs{BS}={exp2[128]['auc_roc']:.4f} | 96 bs{BS_LARGE}={m96l['auc_roc']:.4f}", flush=True)
    if RUN_WANDB:
        run.log({"exp4/auc_128_smallbs": exp2[128]["auc_roc"], "exp4/auc_96_largebs": m96l["auc_roc"]}, step=4)


## 9. Kết luận + lưu TẤT CẢ vào Drive

In [ ]:
import csv as _csv

report = {"exp1": exp1, "exp2": exp2, "exp3": exp3, "exp4": exp4,
          "config": {"epochs": EPOCHS, "bs": BS, "bs_large": BS_LARGE, "train_n": TRAIN_N, "val_n": VAL_N}}
checks = {}
if exp1:
    checks["exp1_ssim>0.85"] = bool(exp1["ssim"] > 0.85)
    checks["exp1_psnr>30"] = bool(exp1["psnr"] > 30.0)
if 96 in exp2 and 128 in exp2:
    checks["exp2_auc_drop<1.5%"] = bool((exp2[128]["auc_roc"] - exp2[96]["auc_roc"]) < 0.015)
if exp3:
    checks["exp3_cka>0.85"] = bool(exp3["cka"] > 0.85)
    checks["exp3_sil_drop<10%"] = bool(exp3["sil_drop"] < 0.10)
if exp4:
    checks["exp4_96largebs>=128smallbs"] = bool(
        exp4[f"96_bs{BS_LARGE}"]["auc_roc"] >= exp4[f"128_bs{BS}"]["auc_roc"] - 0.005)
passed = sum(1 for v in checks.values() if v)
best = 96 if (checks.get("exp4_96largebs>=128smallbs", False) and passed >= max(3, len(checks) - 1)) else 128
report["checks"] = checks
report["passed"] = f"{passed}/{len(checks)}"
report["best"] = f"{best}^3"
print("===== KET LUAN =====")
for k, v in checks.items():
    print(("  PASS " if v else "  FAIL ") + k)
print(f"=> DATASET TOT HON: {best}^3 ({report['passed']})")

report_path = os.path.join(FIG_DIR, "resolution_hf_report.json")
with open(report_path, "w") as fh:
    json.dump(report, fh, indent=2, default=float)

csv_path = os.path.join(FIG_DIR, "resolution_hf_metrics.csv")
with open(csv_path, "w", newline="") as fh:
    w = _csv.writer(fh)
    w.writerow(["experiment", "size", "auc_roc", "auc_pr", "f1", "balanced_acc", "ece", "train_min"])
    for s in SIZES:
        if s in exp2:
            m = exp2[s]
            w.writerow(["exp2", f"{s}^3", m["auc_roc"], m["auc_pr"], m["f1"], m["balanced_acc"], m["ece"], m["train_min"]])
    for k, m in exp4.items():
        w.writerow(["exp4", k, m["auc_roc"], m["auc_pr"], m["f1"], m["balanced_acc"], m["ece"], m["train_min"]])

summary_md = [
    "# Kết quả so sánh 96^3 vs 128^3 (Harvard-GF, classification)",
    "",
    f"- Cấu hình: {EPOCHS} epochs, bs {BS} / bs_large {BS_LARGE}, train_n {TRAIN_N or 'all'}, val_n {VAL_N or 'all'}",
    "",
    "## Chỉ số (Exp2)",
    "| Size | AUC-ROC | AUC-PR | F1 | Balanced Acc | ECE |",
    "|---|---|---|---|---|---|",
]
for s in SIZES:
    if s in exp2:
        m = exp2[s]
        summary_md.append(f"| {s}^3 | {m['auc_roc']:.4f} | {m['auc_pr']:.4f} | {m['f1']:.4f} | {m['balanced_acc']:.4f} | {m['ece']:.4f} |")
summary_md += ["", "## Kiểm tra ngưỡng", "| Check | Pass |", "|---|---|"]
for k, v in checks.items():
    summary_md.append(f"| {k} | {'YES' if v else 'NO'} |")
summary_md += ["", f"**Dataset tốt hơn: {best}^3 ({report['passed']})**", "",
               "## Ghi chú (điền tay)", "- ", ""]
summary_path = os.path.join(FIG_DIR, "RESULT_96_vs_128.md")
with open(summary_path, "w", encoding="utf-8") as fh:
    fh.write("\n".join(summary_md))
print("wrote", report_path, "|", csv_path, "|", summary_path)

if RUN_WANDB:
    run.summary.update({"best": f"{best}^3", "passed": report["passed"]})
    for k, v in checks.items():
        run.summary.update({f"check/{k}": bool(v)})

if DRIVE_READY:
    import shutil
    os.makedirs(DRIVE_FIG, exist_ok=True)
    os.makedirs(DRIVE_MODEL, exist_ok=True)
    for f in os.listdir(FIG_DIR):
        shutil.copy2(os.path.join(FIG_DIR, f), os.path.join(DRIVE_FIG, f))
    for s, model in models.items():
        torch.save({"state_dict": model.state_dict(), "size": s}, os.path.join(DRIVE_MODEL, f"proxy_{s}.pt"))
    print("[drive] figures ->", DRIVE_FIG)
    print("[drive] models  ->", DRIVE_MODEL)
else:
    print("[drive] SKIP (không mount được)")

if RUN_WANDB:
    run.finish()
print("done")


## 10. Ghi chú kết quả (điền tay)

Dataset tốt hơn: `____`³

Lý do / nhận xét:

-

Kết quả đã lưu:
- `resolution_96_128_hf_figures/RESULT_96_vs_128.md`
- `resolution_96_128_hf_figures/resolution_hf_report.json`
- `resolution_96_128_hf_figures/resolution_hf_metrics.csv`
- `resolution_96_128_hf/proxy_96.pt`, `proxy_128.pt`
